# 04_01 - Base SER depurada a escala barrio

Este notebook construye la base nuclear de eventos SER del TFM a escala barrio. Parte de `final_clean_parts`, aplica validaciones cruzadas de calendario y duracion normativa, y prepara una tabla de tiques depurada lista para construir despues `SER_barrio_intervalo`.

El diagnostico de parquimetros se conserva como evidencia metodologica: los tiques con identificador fisico pueden enlazar con inventario de parquimetros, pero los pagos app/canal digital no deben imputarse a un parquimetro o calle con las fuentes publicas disponibles. Por tanto, el core principal pasa a ser barrio; calle/parquimetro queda para un notebook complementario posterior.


## 0. Configuracion inicial

Se detecta la raiz del repositorio a partir de `data_catalog.csv`, se fijan parametros metodologicos y se definen funciones auxiliares de lectura ligera. Las escrituras de produccion quedan desactivadas por defecto en esta iteracion.


In [1]:
from __future__ import annotations

import re
import shutil
import unicodedata
from pathlib import Path
from typing import Iterable

import pandas as pd

try:
    import pyarrow.dataset as ds
    import pyarrow.parquet as pq
except ImportError as exc:
    raise ImportError(
        "Este notebook requiere pyarrow para inspeccionar metadatos parquet sin cargar tablas completas."
    ) from exc

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 30)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 220)

EXECUTE_HEAVY_STEPS = False
WRITE_PRODUCTION_OUTPUTS = False
PROCESS_TICKETS_FULL = False
SAMPLE_STRATEGY = "largest_part_per_period"
OVERWRITE_OUTPUTS = False
MAX_PARTS_FULL_RUN = None
SMOKE_TEST_N_PARTS = None

# Salida temporal opcional para futuras ejecuciones completas. No se usa mientras WRITE_CALENDAR_DURATION_TEMP_OUTPUT=False.
WRITE_CALENDAR_DURATION_TEMP_OUTPUT = False
CALENDAR_DURATION_TEMP_DIR = None

INTERSECTION_RADIUS_M = 15
MAX_CALLES_UNIDAD = 3
DURATION_TOLERANCE_MIN = 5

SER_DURATION_LIMITS_MIN = {
    "VERDE": 120,
    "AZUL": 240,
    "ALTA ROTACION": 45,
    "USO DISUASORIO": 720,
    "AZUL SANITARIA": 240,
    "COMERCIALES": 480,
    "TALLERES": 300,
}

FINAL_BARRIO_TICKETS_COLUMNS = [
    "matricula_parquimetro", "fecha_inicio", "fecha_fin", "fecha", "anio", "mes",
    "dia_semana_num", "hora_inicio", "periodo_hora", "duracion_minutos",
    "cod_distrito", "distrito", "cod_barrio", "barrio", "barrio_key",
    "cod_barrio_compuesto", "tipo_zona", "importe_tique", "tipo_identificador_ser",
]

FINAL_BARRIO_CAPACITY_COLUMNS = [
    "anio", "cod_distrito", "cod_barrio", "barrio", "barrio_key",
    "cod_barrio_compuesto", "plazas_barrio_anio", "n_calles_barrio_anio",
]

FINAL_BARRIO_CAPACITY_OPTIONAL_COLOR_COLUMNS: list[str] = []


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "data_catalog.csv").exists():
            return candidate
    raise FileNotFoundError("No se ha encontrado data_catalog.csv subiendo desde el directorio actual.")


ROOT = find_project_root()
DATA_CATALOG = ROOT / "data_catalog.csv"
REPORTS_TABLES = ROOT / "reports" / "tables"


def relpath(path: Path | str) -> str:
    path = Path(path)
    try:
        return str(path.resolve().relative_to(ROOT))
    except ValueError:
        return str(path)


def normalize_text(value: object) -> str | pd.NA:
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().upper()
    text = "".join(
        ch for ch in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(ch)
    )
    text = re.sub(r"\s+", " ", text)
    return text or pd.NA


def normalize_street(value: object) -> str | pd.NA:
    text = normalize_text(value)
    if pd.isna(text):
        return pd.NA
    text = re.sub(r"^(CALLE|CL|C/|AVENIDA|AVDA|PASEO|Pº|PLAZA|PZA|GLORIETA|GTA)\s+", "", text)
    return re.sub(r"\s+", " ", text).strip() or pd.NA


def safe_parquet_metadata(path: Path) -> dict[str, object]:
    if not path.exists():
        return {
            "path": relpath(path), "exists": False, "n_rows_metadata": pd.NA,
            "n_columns_metadata": pd.NA, "columns": [], "read_error": "file_not_found",
        }
    try:
        pf = pq.ParquetFile(path)
        return {
            "path": relpath(path), "exists": True,
            "n_rows_metadata": int(pf.metadata.num_rows),
            "n_columns_metadata": int(pf.metadata.num_columns),
            "columns": pf.schema_arrow.names,
            "read_error": pd.NA,
        }
    except Exception as exc:
        return {
            "path": relpath(path), "exists": True, "n_rows_metadata": pd.NA,
            "n_columns_metadata": pd.NA, "columns": [],
            "read_error": f"{type(exc).__name__}: {exc}",
        }


def read_parquet_sample_safe(path: Path, columns: list[str] | None = None, n_rows: int = 5) -> tuple[pd.DataFrame, str | pd.NA]:
    try:
        df = pd.read_parquet(path, columns=columns)
        return df.head(n_rows).copy(), pd.NA
    except Exception as exc:
        return pd.DataFrame(), f"{type(exc).__name__}: {exc}"


def require_columns(columns: Iterable[str], required: Iterable[str], source_name: str) -> pd.DataFrame:
    present = set(columns)
    return pd.DataFrame(
        {"dataset_id": source_name, "column": col, "present": col in present}
        for col in required
    )

print("Directorio actual del kernel:", Path.cwd())
print("ROOT detectado:", ROOT)
print("Catalogo existe:", DATA_CATALOG.exists())
print("Ejecucion pesada activada:", EXECUTE_HEAVY_STEPS)
print("Escritura de outputs activada:", WRITE_PRODUCTION_OUTPUTS)
print("Procesamiento completo de tiques activado:", PROCESS_TICKETS_FULL)
print("Estrategia de muestra tecnica:", SAMPLE_STRATEGY)
print("Sobrescritura de outputs activada:", OVERWRITE_OUTPUTS)
print("Limite de partes para ejecucion completa:", MAX_PARTS_FULL_RUN)
print("Smoke test partes:", SMOKE_TEST_N_PARTS)


Directorio actual del kernel: /Users/hugo/TFM_parking_madrid/notebooks
ROOT detectado: /Users/hugo/TFM_parking_madrid
Catalogo existe: True
Ejecucion pesada activada: True
Escritura de outputs activada: True
Procesamiento completo de tiques activado: True
Estrategia de muestra tecnica: largest_part_per_period
Sobrescritura de outputs activada: False
Limite de partes para ejecucion completa: None
Smoke test partes: None


**Lectura/decisión.** El notebook queda parametrizado para ejecución reproducible y conservadora. Las banderas de producción se dejan desactivadas por defecto (`PROCESS_TICKETS_FULL=False`, `WRITE_PRODUCTION_OUTPUTS=False` y `OVERWRITE_OUTPUTS=False`) para evitar reejecuciones accidentales sobre 150 millones de tiques. La ejecución completa ya se realizó de forma controlada y sus salidas fueron validadas posteriormente; cualquier nueva regeneración deberá activarse explícitamente.


## 1. Objetivo y contrato metodologico

El objetivo es construir una base SER depurada a escala barrio, lista para construir despues `SER_barrio_intervalo`. La tabla conserva todos los tiques que superan limpieza individual previa, regimen SER observable y duracion normativa, sin forzar una asignacion a parquimetro fisico cuando el identificador procede de app o canal digital.

Contrato cerrado:

- El filtro de calendario sera estricto: un tique cuya `fecha_inicio` quede fuera del regimen SER observable no entrara en la base barrio.
- Las duraciones se resolveran por `tipo_zona` con limite normativo y tolerancia de 5 minutos.
- La clave espacial principal del core sera barrio: `cod_distrito`, `cod_barrio`, `cod_barrio_compuesto` y `barrio_key`.
- El diagnostico de parquimetros se conserva como evidencia metodologica, pero no define el core principal.
- No se imputaran pagos app/canal digital a calle o parquimetro.
- La escala calle/parquimetro queda como analisis complementario parcial posterior para tiques con identificador fisico.
- Las columnas auxiliares de validacion no permaneceran en las salidas finales si solo sirven para diagnostico.


In [2]:
contract_tables = {
    "regimen_ser_observable": pd.DataFrame([
        {"caso": "lunes-viernes no festivos", "ventana": "09:00-21:00", "entra_en_base": True},
        {"caso": "sabados no festivos", "ventana": "09:00-15:00", "entra_en_base": True},
        {"caso": "agosto lunes-sabado no festivo", "ventana": "09:00-15:00", "entra_en_base": True},
        {"caso": "24 y 31 de diciembre", "ventana": "09:00-15:00", "entra_en_base": True},
        {"caso": "domingos y festivos", "ventana": "sin servicio", "entra_en_base": False},
    ]),
    "limites_duracion": pd.DataFrame([
        {"tipo_zona": tipo, "limite_minutos": limite, "tolerancia_minutos": DURATION_TOLERANCE_MIN}
        for tipo, limite in SER_DURATION_LIMITS_MIN.items()
    ]).sort_values("tipo_zona"),
    "unidad_espacial_core": pd.DataFrame([
        {"decision": "core principal", "unidad": "barrio"},
        {"decision": "analisis complementario posterior", "unidad": "calle/parquimetro para tiques fisicos"},
        {"decision": "no permitido en core", "unidad": "imputar pagos app a parquimetro o calle"},
    ]),
}

for name, table in contract_tables.items():
    print(name)
    display(table)


regimen_ser_observable


,caso,ventana,entra_en_base
0,lunes-viernes no festivos,09:00-21:00,True
1,sabados no festivos,09:00-15:00,True
2,agosto lunes-sabado no festivo,09:00-15:00,True
3,24 y 31 de diciembre,09:00-15:00,True
4,domingos y festivos,sin servicio,False


limites_duracion


,tipo_zona,limite_minutos,tolerancia_minutos
2,ALTA ROTACION,45,5
1,AZUL,240,5
4,AZUL SANITARIA,240,5
5,COMERCIALES,480,5
6,TALLERES,300,5
3,USO DISUASORIO,720,5
0,VERDE,120,5


unidad_espacial_core


,decision,unidad
0,core principal,barrio
1,analisis complementario posterior,calle/parquimetro para tiques fisicos
2,no permitido en core,imputar pagos app a parquimetro o calle


**Lectura/decision.** Las reglas anteriores son parte del contrato del notebook. Los siguientes bloques implementaran estas reglas de forma trazable; por ahora quedan expresadas como tablas internas de decision, no como outputs externos.


## 2. Fuentes de entrada y salidas previstas

Las fuentes se localizan desde `data_catalog.csv` y desde los manifiestos ya existentes de tiques limpios por partes. Las salidas previstas son Parquet de produccion en `data/processed/core/ser/`.


In [3]:
INPUT_DATASET_IDS = [
    "ser_tiques",
    "contexto_calendario_laboral",
    "ser_parquimetros",
    "ser_calles_plazas",
]
OPTIONAL_INPUT_DATASET_IDS = ["callejero_viales_vigentes"]

TIQUES_FINAL_PARTS_DIR = ROOT / "data" / "interim" / "ser" / "ser_tiques" / "final_clean_parts"
TIQUES_BASE_PARTS_DIR = ROOT / "data" / "interim" / "ser" / "ser_tiques" / "base_clean_parts"

OUTPUT_TICKETS_BARRIO_DIR = ROOT / "data" / "processed" / "core" / "ser" / "ser_tiques_barrio_base"
OUTPUT_BARRIO_CAPACITY_PATH = ROOT / "data" / "processed" / "core" / "ser" / "ser_barrio_capacidad_anio.parquet"

catalog = pd.read_csv(DATA_CATALOG)
source_catalog = catalog[catalog["dataset_id"].isin(INPUT_DATASET_IDS + OPTIONAL_INPUT_DATASET_IDS)].copy()

operational_source_decisions = pd.DataFrame([
    {
        "fuente": "ser_tiques",
        "rol_04_01": "entrada_core",
        "ruta_operativa": relpath(TIQUES_FINAL_PARTS_DIR),
        "decision": "leer por particiones desde final_clean_parts",
    },
    {
        "fuente": "contexto_calendario_laboral",
        "rol_04_01": "entrada_core",
        "ruta_operativa": relpath(ROOT / "data" / "interim" / "contexto" / "contexto_calendario_laboral" / "contexto_calendario_laboral_clean.parquet"),
        "decision": "filtrar regimen SER observable",
    },
    {
        "fuente": "ser_calles_plazas",
        "rol_04_01": "entrada_core",
        "ruta_operativa": relpath(ROOT / "data" / "interim" / "ser" / "ser_calles_plazas" / "ser_calles_plazas_clean.parquet"),
        "decision": "calcular capacidad anual por barrio",
    },
    {
        "fuente": "ser_parquimetros",
        "rol_04_01": "diagnostico_metodologico",
        "ruta_operativa": relpath(ROOT / "data" / "interim" / "ser" / "ser_parquimetros" / "ser_parquimetros_clean.parquet"),
        "decision": "diagnosticar app/fisico; no define el core barrio",
    },
    {
        "fuente": "callejero_viales_vigentes",
        "rol_04_01": "opcional_no_usado_core",
        "ruta_operativa": relpath(ROOT / "data" / "interim" / "cartografia" / "callejero_viales_vigentes" / "callejero_viales_vigentes_clean.parquet"),
        "decision": "reservado para analisis calle/parquimetro posterior",
    },
])

tiques_source_decision = pd.DataFrame([
    {
        "nivel": "final_clean_parts",
        "path": relpath(TIQUES_FINAL_PARTS_DIR),
        "uso_04_01": "fuente operativa principal",
        "motivo": "salida limpia final de 02_01; evita rehacer limpieza individual",
        "existe": TIQUES_FINAL_PARTS_DIR.exists(),
    },
    {
        "nivel": "base_clean_parts",
        "path": relpath(TIQUES_BASE_PARTS_DIR),
        "uso_04_01": "no se usa",
        "motivo": "nivel previo de limpieza; no es fuente operativa del core barrio",
        "existe": TIQUES_BASE_PARTS_DIR.exists(),
    },
])

outputs_planned = pd.DataFrame([
    {"output": "ser_tiques_barrio_base", "path": relpath(OUTPUT_TICKETS_BARRIO_DIR), "tipo": "dataset parquet particionable"},
    {"output": "ser_barrio_capacidad_anio", "path": relpath(OUTPUT_BARRIO_CAPACITY_PATH), "tipo": "parquet"},
])

print("Directorio final_clean_parts existe:", TIQUES_FINAL_PARTS_DIR.exists())
print("Directorio base_clean_parts existe, pero no se usa en 04_01:", TIQUES_BASE_PARTS_DIR.exists())
display(operational_source_decisions)
display(tiques_source_decision)
display(outputs_planned)


Directorio final_clean_parts existe: True
Directorio base_clean_parts existe, pero no se usa en 04_01: True


,fuente,rol_04_01,ruta_operativa,decision
0,ser_tiques,entrada_core,data/interim/ser/ser_tiques/final_clean_parts,leer por particiones desde final_clean_parts
1,contexto_calendario_laboral,entrada_core,data/interim/contexto/contexto_calendario_laboral/contexto_calendario_laboral_clean.parquet,filtrar regimen SER observable
2,ser_calles_plazas,entrada_core,data/interim/ser/ser_calles_plazas/ser_calles_plazas_clean.parquet,calcular capacidad anual por barrio
3,ser_parquimetros,diagnostico_metodologico,data/interim/ser/ser_parquimetros/ser_parquimetros_clean.parquet,diagnosticar app/fisico; no define el core barrio
4,callejero_viales_vigentes,opcional_no_usado_core,data/interim/cartografia/callejero_viales_vigentes/callejero_viales_vigentes_clean.parquet,reservado para analisis calle/parquimetro posterior


,nivel,path,uso_04_01,motivo,existe
0,final_clean_parts,data/interim/ser/ser_tiques/final_clean_parts,fuente operativa principal,salida limpia final de 02_01; evita rehacer limpieza individual,True
1,base_clean_parts,data/interim/ser/ser_tiques/base_clean_parts,no se usa,nivel previo de limpieza; no es fuente operativa del core barrio,True


,output,path,tipo
0,ser_tiques_barrio_base,data/processed/core/ser/ser_tiques_barrio_base,dataset parquet particionable
1,ser_barrio_capacidad_anio,data/processed/core/ser/ser_barrio_capacidad_anio.parquet,parquet


**Lectura/decision.** Las rutas de entrada y salida quedan declaradas sin crear nuevos reports ni modificar `data/raw` o `data/interim`. La tabla visible muestra decisiones operativas, no el `archivo_interim` generico del catalogo: los tiques se leen desde `final_clean_parts`, calendario y `ser_calles_plazas` son entradas core, `ser_parquimetros` queda como diagnostico metodologico y callejero se reserva para un analisis posterior fuera del core barrio.


## 3. Carga ligera y validacion minima de fuentes limpias

Se inspeccionan metadatos Parquet y manifiestos sin cargar tablas completas. Para tiques se trabaja desde el manifiesto de partes, de modo que las futuras transformaciones puedan procesarse por particion o chunk.


In [4]:
def extract_periodo_inicio_from_path(path: Path) -> str | pd.NA:
    for part in path.parts:
        if part.startswith("periodo_inicio="):
            return part.split("=", 1)[1]
    return pd.NA


def load_tiques_final_parts_manifest() -> pd.DataFrame:
    if not TIQUES_FINAL_PARTS_DIR.exists():
        raise FileNotFoundError(f"No existe el directorio de tiques finales: {TIQUES_FINAL_PARTS_DIR}")

    rows = []
    for path in sorted(TIQUES_FINAL_PARTS_DIR.glob("**/*.parquet")):
        meta = safe_parquet_metadata(path)
        rows.append({
            "periodo_inicio": extract_periodo_inicio_from_path(path),
            "path": meta["path"],
            "path_abspath": path,
            "n_rows": meta["n_rows_metadata"],
            "n_columns": meta["n_columns_metadata"],
            "output_size_mb": round(path.stat().st_size / (1024**2), 3),
            "read_error": meta["read_error"],
        })

    manifest = pd.DataFrame(rows)
    if manifest.empty:
        raise FileNotFoundError(f"No se han encontrado Parquet bajo {TIQUES_FINAL_PARTS_DIR}")
    return manifest


def iter_tique_final_part_paths(manifest: pd.DataFrame) -> Iterable[Path]:
    for path in manifest["path_abspath"]:
        yield Path(path)


tiques_final_manifest = load_tiques_final_parts_manifest()
expected_tiques_final_columns = [
    "matricula_parquimetro", "fecha_inicio", "fecha_fin", "duracion_minutos",
    "cod_distrito", "distrito", "cod_barrio", "barrio", "tipo_zona", "importe_tique",
]

first_final_part = next(iter_tique_final_part_paths(tiques_final_manifest))
first_final_meta = safe_parquet_metadata(first_final_part)
missing_tiques_final_columns = [
    col for col in expected_tiques_final_columns
    if col not in first_final_meta["columns"]
]
extra_tiques_final_columns = [
    col for col in first_final_meta["columns"]
    if col not in expected_tiques_final_columns
]

manifest_summary = pd.DataFrame([{
    "fuente_tiques": relpath(TIQUES_FINAL_PARTS_DIR),
    "n_parts_finales": int(len(tiques_final_manifest)),
    "n_rows_finales": int(tiques_final_manifest["n_rows"].sum()),
    "n_columns": ", ".join(map(str, sorted(tiques_final_manifest["n_columns"].dropna().unique()))),
    "periodos_inicio_presentes": ", ".join(map(str, sorted(tiques_final_manifest["periodo_inicio"].dropna().unique()))),
    "columnas_esperadas": ", ".join(expected_tiques_final_columns),
    "columnas_faltantes": ", ".join(missing_tiques_final_columns) if missing_tiques_final_columns else "",
    "columnas_extra": ", ".join(extra_tiques_final_columns) if extra_tiques_final_columns else "",
    "partes_con_error_metadatos": int(tiques_final_manifest["read_error"].notna().sum()),
}])

clean_paths = {
    "contexto_calendario_laboral": ROOT / "data" / "interim" / "contexto" / "contexto_calendario_laboral" / "contexto_calendario_laboral_clean.parquet",
    "ser_parquimetros": ROOT / "data" / "interim" / "ser" / "ser_parquimetros" / "ser_parquimetros_clean.parquet",
    "ser_calles_plazas": ROOT / "data" / "interim" / "ser" / "ser_calles_plazas" / "ser_calles_plazas_clean.parquet",
    "callejero_viales_vigentes": ROOT / "data" / "interim" / "cartografia" / "callejero_viales_vigentes" / "callejero_viales_vigentes_clean.parquet",
}

metadata_rows = []
for dataset_id, path in clean_paths.items():
    meta = safe_parquet_metadata(path)
    metadata_rows.append({
        "dataset_id": dataset_id,
        "path": meta["path"],
        "exists": meta["exists"],
        "n_rows_metadata": meta["n_rows_metadata"],
        "n_columns_metadata": meta["n_columns_metadata"],
        "read_error": meta["read_error"],
    })

source_metadata = pd.DataFrame(metadata_rows)
source_role_map = {
    "contexto_calendario_laboral": "entrada_core",
    "ser_calles_plazas": "entrada_core",
    "ser_tiques": "entrada_core",
    "ser_parquimetros": "diagnostico_metodologico",
    "callejero_viales_vigentes": "opcional_no_usado_core",
}
source_metadata["papel_core_barrio"] = source_metadata["dataset_id"].map(source_role_map).fillna("sin_clasificar")

display(manifest_summary)
display(tiques_final_manifest.groupby("periodo_inicio", as_index=False).agg(
    n_parts=("path", "size"),
    n_rows=("n_rows", "sum"),
    n_columns=("n_columns", "max"),
))
display(source_metadata)


,fuente_tiques,n_parts_finales,n_rows_finales,n_columns,periodos_inicio_presentes,columnas_esperadas,columnas_faltantes,columnas_extra,partes_con_error_metadatos
0,data/interim/ser/ser_tiques/final_clean_parts,1002,150137884,10,"2023_q1, 2023_q2, 2023_q3, 2023_q4, 2024_q1, 2024_q2, 2024_q3, 2024_q4, 2025_q1, 2025_q2, 2025_q3, 2025_q4, 2026_q1","matricula_parquimetro, fecha_inicio, fecha_fin, duracion_minutos, cod_distrito, distrito, cod_barrio, barrio, tipo_zona, importe_tique",,,0


,periodo_inicio,n_parts,n_rows,n_columns
0,2023_q1,48,11907716,10
1,2023_q2,94,11665684,10
2,2023_q3,63,9651162,10
3,2023_q4,86,11560293,10
4,2024_q1,96,12104937,10
5,2024_q2,99,12490394,10
6,2024_q3,82,7910433,10
7,2024_q4,68,12593853,10
8,2025_q1,102,12749010,10
9,2025_q2,68,12361188,10


,dataset_id,path,exists,n_rows_metadata,n_columns_metadata,read_error,papel_core_barrio
0,contexto_calendario_laboral,data/interim/contexto/contexto_calendario_laboral/contexto_calendario_laboral_clean.parquet,True,1461,17,<NA>,entrada_core
1,ser_parquimetros,data/interim/ser/ser_parquimetros/ser_parquimetros_clean.parquet,True,4772,14,<NA>,diagnostico_metodologico
2,ser_calles_plazas,data/interim/ser/ser_calles_plazas/ser_calles_plazas_clean.parquet,True,134909,12,<NA>,entrada_core
3,callejero_viales_vigentes,data/interim/cartografia/callejero_viales_vigentes/callejero_viales_vigentes_clean.parquet,True,9287,6,<NA>,opcional_no_usado_core


**Lectura/decision.** La validacion ligera se basa en metadatos y existencia fisica. El manifiesto interno de tiques se construye en memoria desde los Parquet existentes en `final_clean_parts`, sin escribir CSVs nuevos. Esta es la salida limpia individual de `02_01`; `04_01` parte de ella y se limita a resolver validaciones cruzadas pendientes.


In [5]:
required_columns = {
    "contexto_calendario_laboral": ["fecha", "anio", "mes", "dia_semana_num", "es_festivo", "es_domingo", "es_sabado"],
    "ser_parquimetros": ["matricula", "fecha_de_alta", "fecha_de_baja", "cod_barrio", "barrio", "calle", "longitud", "latitud"],
    "ser_calles_plazas": ["anio", "cod_barrio", "barrio", "calle", "color", "numero_plazas"],
    "callejero_viales_vigentes": ["nombre_via_completo", "geometry"],
}

column_check_rows = []
for dataset_id, path in clean_paths.items():
    meta = safe_parquet_metadata(path)
    column_check_rows.append(require_columns(meta["columns"], required_columns[dataset_id], dataset_id))

column_checks = pd.concat(column_check_rows, ignore_index=True)
column_checks_compact = (
    column_checks.groupby("dataset_id", as_index=False)
    .agg(
        n_columnas_requeridas=("column", "size"),
        n_columnas_presentes=("present", "sum"),
    )
)
column_checks_compact["n_columnas_faltantes"] = (
    column_checks_compact["n_columnas_requeridas"] - column_checks_compact["n_columnas_presentes"]
)
missing_columns = column_checks.loc[~column_checks["present"], ["dataset_id", "column"]]

display(column_checks_compact)
if missing_columns.empty:
    print("No hay columnas requeridas faltantes en las fuentes limpias inspeccionadas.")
else:
    display(missing_columns)


,dataset_id,n_columnas_requeridas,n_columnas_presentes,n_columnas_faltantes
0,callejero_viales_vigentes,2,2,0
1,contexto_calendario_laboral,7,7,0
2,ser_calles_plazas,6,6,0
3,ser_parquimetros,8,8,0


No hay columnas requeridas faltantes en las fuentes limpias inspeccionadas.


**Lectura/decision.** El control compacto confirma si las columnas contractuales estan disponibles. `callejero_viales_vigentes` se mantiene como fuente opcional no usada por el core barrio; no bloquea esta iteracion.


## 4. Reglas de regimen SER observable

El filtro de calendario se implementara a partir de `fecha_inicio` y calendario laboral limpio. La decision sera estricta: los tiques fuera de ventana observable se eliminan de la base join.


In [6]:
def load_calendar_ser_observable(path: Path) -> pd.DataFrame:
    columns = ["fecha", "anio", "mes", "dia_semana_num", "es_festivo", "es_sabado", "es_domingo"]
    calendar = pd.read_parquet(path, columns=columns).copy()
    calendar["fecha"] = pd.to_datetime(calendar["fecha"]).dt.normalize()

    bool_cols = ["es_festivo", "es_sabado", "es_domingo"]
    for col in bool_cols:
        calendar[col] = calendar[col].fillna(False).astype(bool)

    return calendar.drop_duplicates("fecha")


def add_time_fields(tickets: pd.DataFrame) -> pd.DataFrame:
    out = tickets.copy()
    out["fecha_inicio"] = pd.to_datetime(out["fecha_inicio"], errors="coerce")
    out["fecha_fin"] = pd.to_datetime(out["fecha_fin"], errors="coerce")
    out["fecha"] = out["fecha_inicio"].dt.normalize()
    out["anio"] = out["fecha_inicio"].dt.year.astype("Int64")
    out["mes"] = out["fecha_inicio"].dt.month.astype("Int64")
    out["dia_semana_num"] = out["fecha_inicio"].dt.weekday.astype("Int64") + 1
    out["hora_inicio"] = out["fecha_inicio"].dt.time
    out["periodo_hora"] = out["fecha_inicio"].dt.floor("h")
    return out


def filter_ser_observable_calendar(tickets: pd.DataFrame, calendar: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int | float]]:
    work = add_time_fields(tickets)
    n_in = int(len(work))

    calendar_join = calendar[["fecha", "es_festivo", "es_sabado", "es_domingo"]].copy()
    calendar_join["fecha_en_calendario"] = True

    work = work.merge(
        calendar_join,
        on="fecha",
        how="left",
        validate="many_to_one",
    )

    fecha_en_calendario = work["fecha_en_calendario"].fillna(False).astype(bool)

    for col in ["es_festivo", "es_sabado", "es_domingo"]:
        work[col] = work[col].fillna(False).astype(bool)

    work["fecha_en_calendario"] = fecha_en_calendario

    minutes = work["fecha_inicio"].dt.hour * 60 + work["fecha_inicio"].dt.minute
    is_dec_24_31 = (work["mes"] == 12) & (work["fecha_inicio"].dt.day.isin([24, 31]))
    is_august = work["mes"] == 8
    is_saturday = work["dia_semana_num"] == 6
    is_monday_friday = work["dia_semana_num"].between(1, 5)
    is_service_day = fecha_en_calendario & ~(work["es_domingo"] | work["es_festivo"])

    window_lunes_viernes = is_service_day & is_monday_friday & ~is_august & ~is_dec_24_31
    window_sabado = is_service_day & is_saturday & ~is_august & ~is_dec_24_31
    window_agosto = is_service_day & is_august & (is_monday_friday | is_saturday)
    window_24_31_dic = is_service_day & is_dec_24_31 & (is_monday_friday | is_saturday)

    in_long_window = window_lunes_viernes & minutes.between(9 * 60, 21 * 60 - 1)
    in_short_saturday = window_sabado & minutes.between(9 * 60, 15 * 60 - 1)
    in_short_august = window_agosto & minutes.between(9 * 60, 15 * 60 - 1)
    in_short_dec = window_24_31_dic & minutes.between(9 * 60, 15 * 60 - 1)
    observable = in_long_window | in_short_saturday | in_short_august | in_short_dec

    fecha_sin_calendario = ~fecha_en_calendario
    fuera_dia_servicio = fecha_en_calendario & ~is_service_day
    fuera_horario_lunes_viernes = window_lunes_viernes & ~in_long_window
    fuera_horario_sabado = window_sabado & ~in_short_saturday
    fuera_horario_agosto = window_agosto & ~in_short_august
    fuera_horario_24_31_dic = window_24_31_dic & ~in_short_dec

    summary = {
        "n_tiques_entrada": n_in,
        "n_fecha_sin_calendario": int(fecha_sin_calendario.sum()),
        "n_fuera_dia_servicio": int(fuera_dia_servicio.sum()),
        "n_fuera_horario_lunes_viernes": int(fuera_horario_lunes_viernes.sum()),
        "n_fuera_horario_sabado": int(fuera_horario_sabado.sum()),
        "n_fuera_horario_agosto": int(fuera_horario_agosto.sum()),
        "n_fuera_horario_24_31_dic": int(fuera_horario_24_31_dic.sum()),
        "n_tiques_salida": int(observable.sum()),
    }
    summary["pct_eliminado"] = round((1 - summary["n_tiques_salida"] / n_in) * 100, 3) if n_in else 0.0

    drop_aux = ["fecha_en_calendario", "es_festivo", "es_sabado", "es_domingo"]
    filtered = work.loc[observable].drop(columns=drop_aux).reset_index(drop=True)
    return filtered, summary


def selected_tique_parts_for_light_outputs(manifest: pd.DataFrame) -> pd.DataFrame:
    return (
        manifest.sort_values(
            ["periodo_inicio", "n_rows", "path"],
            ascending=[True, False, True],
        )
        .groupby("periodo_inicio", as_index=False, sort=True)
        .head(1)
        .reset_index(drop=True)
        .copy()
    )


calendar_ser = load_calendar_ser_observable(clean_paths["contexto_calendario_laboral"])
tiques_parts_to_process = selected_tique_parts_for_light_outputs(tiques_final_manifest)

calendar_rows = []
calendar_filtered_parts = []
for _, part in tiques_parts_to_process.iterrows():
    tickets_part = pd.read_parquet(part["path_abspath"], columns=expected_tiques_final_columns)
    filtered_part, summary = filter_ser_observable_calendar(tickets_part, calendar_ser)
    filtered_part["periodo_inicio"] = part["periodo_inicio"]
    summary["periodo_inicio"] = part["periodo_inicio"]
    summary["path"] = part["path"]
    calendar_rows.append(summary)
    calendar_filtered_parts.append(filtered_part)

calendar_filter_summary_by_part = pd.DataFrame(calendar_rows)
calendar_filter_summary = pd.DataFrame([{
    "modo": f"{SAMPLE_STRATEGY}_{len(tiques_parts_to_process)}_particiones",
    "n_tiques_entrada": int(calendar_filter_summary_by_part["n_tiques_entrada"].sum()),
    "n_fecha_sin_calendario": int(calendar_filter_summary_by_part["n_fecha_sin_calendario"].sum()),
    "n_fuera_dia_servicio": int(calendar_filter_summary_by_part["n_fuera_dia_servicio"].sum()),
    "n_fuera_horario_lunes_viernes": int(calendar_filter_summary_by_part["n_fuera_horario_lunes_viernes"].sum()),
    "n_fuera_horario_sabado": int(calendar_filter_summary_by_part["n_fuera_horario_sabado"].sum()),
    "n_fuera_horario_agosto": int(calendar_filter_summary_by_part["n_fuera_horario_agosto"].sum()),
    "n_fuera_horario_24_31_dic": int(calendar_filter_summary_by_part["n_fuera_horario_24_31_dic"].sum()),
    "n_tiques_salida": int(calendar_filter_summary_by_part["n_tiques_salida"].sum()),
}])
calendar_filter_summary["pct_eliminado"] = round(
    (1 - calendar_filter_summary["n_tiques_salida"] / calendar_filter_summary["n_tiques_entrada"]) * 100,
    3,
)

tiques_after_calendar_sample = pd.concat(calendar_filtered_parts, ignore_index=True) if calendar_filtered_parts else pd.DataFrame()

display(calendar_filter_summary)
display(calendar_filter_summary_by_part[[
    "periodo_inicio", "n_tiques_entrada", "n_tiques_salida", "pct_eliminado", "path"
]])


,modo,n_tiques_entrada,n_fecha_sin_calendario,n_fuera_dia_servicio,n_fuera_horario_lunes_viernes,n_fuera_horario_sabado,n_fuera_horario_agosto,n_fuera_horario_24_31_dic,n_tiques_salida,pct_eliminado
0,largest_part_per_period_13_particiones,3249757,0,261,0,2,4,90,3249400,0.011


,periodo_inicio,n_tiques_entrada,n_tiques_salida,pct_eliminado,path
0,2023_q1,249988,249728,0.104,data/interim/ser/ser_tiques/final_clean_parts/periodo_inicio=2023_q1/part_000092__raw_2023_q1.parquet
1,2023_q2,250000,250000,0.000,data/interim/ser/ser_tiques/final_clean_parts/periodo_inicio=2023_q2/part_000131__raw_2023_q2.parquet
2,2023_q3,249972,249970,0.001,data/interim/ser/ser_tiques/final_clean_parts/periodo_inicio=2023_q3/part_000226__raw_2023_q3.parquet
3,2023_q4,249957,249956,0.000,data/interim/ser/ser_tiques/final_clean_parts/periodo_inicio=2023_q4/part_000292__raw_2023_q4.parquet
4,2024_q1,249954,249954,0.000,data/interim/ser/ser_tiques/final_clean_parts/periodo_inicio=2024_q1/part_000356__raw_2024_q1.parquet
5,2024_q2,249947,249945,0.001,data/interim/ser/ser_tiques/final_clean_parts/periodo_inicio=2024_q2/part_000500__raw_2024_q2.parquet
6,2024_q3,250000,250000,0.000,data/interim/ser/ser_tiques/final_clean_parts/periodo_inicio=2024_q3/part_000537__raw_2024_q3.parquet
7,2024_q4,249941,249851,0.036,data/interim/ser/ser_tiques/final_clean_parts/periodo_inicio=2024_q4/part_000671__raw_2024_q4.parquet
8,2025_q1,249999,249999,0.000,data/interim/ser/ser_tiques/final_clean_parts/periodo_inicio=2025_q1/part_000694__raw_2025_q1.parquet
9,2025_q2,250000,250000,0.000,data/interim/ser/ser_tiques/final_clean_parts/periodo_inicio=2025_q2/part_000765__raw_2025_q2.parquet


**Lectura/decisión.** Los outputs mostrados corresponden a una muestra técnica controlada que toma la partición de mayor tamaño por `periodo_inicio`. Esta muestra no sustituye la lectura global del histórico, pero permite verificar localmente la implementación del filtro de calendario antes de procesar todas las partes. La validación global del histórico completo se muestra en el bloque 10.

## 5. Resolucion de duraciones por `tipo_zona`

Las duraciones se resolveran con limite normativo y tolerancia de 5 minutos. Los tiques con exceso superior a la tolerancia saldran de la base join.


In [7]:
def resolve_duration_by_zone(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    work = df.copy()
    work["tipo_zona_norm"] = work["tipo_zona"].map(normalize_text)
    limits = pd.DataFrame([
        {"tipo_zona_norm": tipo, "tipo_zona": tipo, "limite_minutos": limite}
        for tipo, limite in SER_DURATION_LIMITS_MIN.items()
    ])

    work = work.merge(
        limits[["tipo_zona_norm", "limite_minutos"]],
        on="tipo_zona_norm",
        how="left",
        validate="many_to_one",
    )

    dur = pd.to_numeric(work["duracion_minutos"], errors="coerce")
    limit = pd.to_numeric(work["limite_minutos"], errors="coerce")
    known_limit = limit.notna()
    to_cap = known_limit & (dur > limit) & (dur <= limit + DURATION_TOLERANCE_MIN)
    to_drop = dur.isna() | (~known_limit) | (dur > limit + DURATION_TOLERANCE_MIN)
    to_keep = ~to_drop

    work.loc[to_cap, "duracion_minutos"] = work.loc[to_cap, "limite_minutos"]

    decision = work.assign(
        _n_entrada=1,
        _n_capados=to_cap.astype(int),
        _n_eliminados=to_drop.astype(int),
        _n_salida=to_keep.astype(int),
    )
    summary = (
        decision.groupby(["tipo_zona_norm", "limite_minutos"], dropna=False, as_index=False)
        .agg(
            n_entrada=("_n_entrada", "sum"),
            n_capados=("_n_capados", "sum"),
            n_eliminados=("_n_eliminados", "sum"),
            n_salida=("_n_salida", "sum"),
        )
        .rename(columns={"tipo_zona_norm": "tipo_zona"})
        .sort_values(["tipo_zona"], na_position="last")
    )
    summary["pct_capado"] = (summary["n_capados"] / summary["n_entrada"] * 100).round(3)
    summary["pct_eliminado"] = (summary["n_eliminados"] / summary["n_entrada"] * 100).round(3)

    filtered = work.loc[to_keep].drop(columns=["tipo_zona_norm", "limite_minutos"]).reset_index(drop=True)
    return filtered, summary



def process_calendar_duration_parts(
    manifest: pd.DataFrame,
    calendar: pd.DataFrame,
    write_temp_output: bool = WRITE_CALENDAR_DURATION_TEMP_OUTPUT,
    temp_dir: Path | None = CALENDAR_DURATION_TEMP_DIR,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Procesa calendario y duracion por particiones sin cargar todo el historico en memoria.

    No se llama por defecto en esta iteracion. Si `write_temp_output=True`, `temp_dir` debe apuntar
    a una ruta explicita fuera de `data/raw` y `data/interim`; esta funcion no escribe produccion.
    """
    if write_temp_output:
        if temp_dir is None:
            raise ValueError("Para escribir salida temporal hay que definir CALENDAR_DURATION_TEMP_DIR.")
        temp_dir = Path(temp_dir)
        temp_dir.mkdir(parents=True, exist_ok=True)

    calendar_rows = []
    duration_rows = []
    for _, part in manifest.iterrows():
        tickets_part = pd.read_parquet(part["path_abspath"], columns=expected_tiques_final_columns)
        filtered_calendar, calendar_summary = filter_ser_observable_calendar(tickets_part, calendar)
        filtered_calendar["periodo_inicio"] = part["periodo_inicio"]
        filtered_duration, duration_summary_part = resolve_duration_by_zone(filtered_calendar)

        calendar_summary["periodo_inicio"] = part["periodo_inicio"]
        calendar_summary["path"] = part["path"]
        calendar_rows.append(calendar_summary)

        duration_summary_part["periodo_inicio"] = part["periodo_inicio"]
        duration_summary_part["path"] = part["path"]
        duration_rows.append(duration_summary_part)

        if write_temp_output:
            rel_part = Path(part["path"]).relative_to(relpath(TIQUES_FINAL_PARTS_DIR))
            out_path = temp_dir / rel_part
            out_path.parent.mkdir(parents=True, exist_ok=True)
            filtered_duration.to_parquet(out_path, index=False)

    return pd.DataFrame(calendar_rows), pd.concat(duration_rows, ignore_index=True) if duration_rows else pd.DataFrame()


tiques_after_duration_sample, duration_summary = resolve_duration_by_zone(tiques_after_calendar_sample)

display(duration_summary[[
    "tipo_zona", "limite_minutos", "n_entrada", "n_capados", "n_eliminados", "n_salida", "pct_capado", "pct_eliminado"
]])


,tipo_zona,limite_minutos,n_entrada,n_capados,n_eliminados,n_salida,pct_capado,pct_eliminado
0,ALTA ROTACION,45,13058,0,371,12687,0.00,2.841
1,AZUL,240,1253480,0,88583,1164897,0.00,7.067
2,AZUL SANITARIA,240,27999,0,836,27163,0.00,2.986
3,COMERCIALES,480,171549,18,0,171549,0.01,0.000
4,TALLERES,300,1289,0,0,1289,0.00,0.000
5,USO DISUASORIO,720,34706,0,1284,33422,0.00,3.700
6,VERDE,120,1747319,0,98442,1648877,0.00,5.634


**Lectura/decisión.** Los porcentajes mostrados corresponden a la muestra técnica de mayor tamaño por `periodo_inicio` y no deben interpretarse como distribución global de excesos de duración. La regla normativa queda implementada sobre `duracion_minutos`: conserva tiques dentro del límite, capa excesos de hasta 5 minutos de tolerancia y elimina duraciones nulas, zonas sin límite o excesos superiores a la tolerancia. La validación global del histórico completo se muestra en el bloque 10.

## 6. Diagnostico del identificador SER y join temporal con parquimetros

Este bloque conserva el join temporal con parquimetros como diagnostico metodologico, no como eje del core. Sirve para distinguir tiques con identificador fisico enlazable de pagos app/canal digital que no deben imputarse a parquimetro o calle con las fuentes publicas disponibles.


In [8]:
def normalize_matricula(value: object) -> str | pd.NA:
    if pd.isna(value):
        return pd.NA
    text = re.sub(r"\s+", "", str(value).strip().upper())
    return text or pd.NA


def normalize_matricula_strip_dot_zero(value: object) -> str | pd.NA:
    text = normalize_matricula(value)
    if pd.isna(text):
        return pd.NA
    text = re.sub(r"\.0$", "", text)
    return text or pd.NA


def normalize_matricula_digits_only(value: object) -> str | pd.NA:
    text = normalize_matricula_strip_dot_zero(value)
    if pd.isna(text):
        return pd.NA
    digits = re.sub(r"\D+", "", text)
    return digits or pd.NA


def normalize_matricula_digits_zfill(value: object, width: int) -> str | pd.NA:
    digits = normalize_matricula_digits_only(value)
    if pd.isna(digits):
        return pd.NA
    return str(digits).zfill(width)


def classify_ser_identifier(value: object) -> str:
    text = normalize_matricula(value)
    if pd.isna(text):
        return "sin_identificador"
    if re.search(r"[A-Z]", str(text)):
        return "canal_app_o_digital"
    return "identificador_fisico_numeric"


def load_parquimetros_view(path: Path) -> pd.DataFrame:
    expected = [
        "matricula", "fecha_de_alta", "fecha_de_baja", "cod_distrito", "distrito",
        "cod_barrio", "num_barrio", "barrio", "calle", "gis_x", "gis_y", "longitud", "latitud",
    ]
    parq = pd.read_parquet(path)
    missing = [col for col in expected if col not in parq.columns]
    if missing:
        raise ValueError(f"Faltan columnas esperadas en ser_parquimetros_clean: {missing}")

    parq = parq[expected].copy()
    parq["matricula_norm"] = parq["matricula"].map(normalize_matricula)
    parq["fecha_de_alta"] = pd.to_datetime(parq["fecha_de_alta"], errors="coerce")
    parq["fecha_de_baja"] = pd.to_datetime(parq["fecha_de_baja"], errors="coerce")
    parq["_parquimetro_row_id"] = range(len(parq))

    return parq.rename(columns={
        "cod_distrito": "cod_distrito_parquimetro",
        "distrito": "distrito_parquimetro",
        "cod_barrio": "cod_barrio_parquimetro",
        "num_barrio": "num_barrio_parquimetro",
        "barrio": "barrio_parquimetro",
    })


def matricula_profile(df: pd.DataFrame, original_col: str, norm_col: str, label: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    original = df[original_col]
    norm = df[norm_col]
    summary = pd.DataFrame([{
        "fuente": label,
        "n_registros": int(len(df)),
        "n_matricula_nula": int(original.isna().sum()),
        "n_matricula_distinta": int(norm.dropna().nunique()),
    }])
    examples = original.dropna().drop_duplicates().head(20).reset_index(drop=True).to_frame("ejemplo_valor_original")
    lengths = (
        norm.dropna().astype(str).str.len().value_counts().sort_index()
        .rename_axis("longitud_matricula_normalizada")
        .reset_index(name="n_registros")
    )
    top = norm.value_counts(dropna=True).head(20).rename_axis("matricula_norm").reset_index(name="n_registros")
    return summary, examples, lengths, top


def compare_matricula_sets(tickets: pd.DataFrame, parquimetros: pd.DataFrame, ticket_norm_col: str, parq_norm_col: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    ticket_norm = tickets[ticket_norm_col]
    parq_norm = parquimetros[parq_norm_col]
    ticket_set = set(ticket_norm.dropna())
    parq_set = set(parq_norm.dropna())
    intersection = ticket_set & parq_set
    ticket_non_null = ticket_norm.notna()
    ticket_in_parq = ticket_non_null & ticket_norm.isin(parq_set)

    summary = pd.DataFrame([{
        "n_matriculas_tiques_distintas": len(ticket_set),
        "n_matriculas_parquimetros_distintas": len(parq_set),
        "n_matriculas_interseccion": len(intersection),
        "pct_matriculas_tiques_en_parquimetros": round(len(intersection) / len(ticket_set) * 100, 3) if ticket_set else 0.0,
        "pct_tiques_con_matricula_en_parquimetros": round(ticket_in_parq.sum() / ticket_non_null.sum() * 100, 3) if ticket_non_null.sum() else 0.0,
    }])

    top_unmatched = (
        ticket_norm.loc[ticket_non_null & ~ticket_norm.isin(parq_set)]
        .value_counts()
        .head(30)
        .rename_axis("matricula_tique_no_presente_en_parquimetros")
        .reset_index(name="n_tiques")
    )
    return summary, top_unmatched


def matricula_variant_diagnostics(tickets: pd.DataFrame, parquimetros: pd.DataFrame) -> pd.DataFrame:
    variants = []
    base = pd.DataFrame({
        "variant": "normalizacion_actual",
        "ticket_norm": tickets["matricula_parquimetro"].map(normalize_matricula),
    })
    parq_base = parquimetros["matricula"].map(normalize_matricula)
    variants.append(("normalizacion_actual", base["ticket_norm"], parq_base))

    ticket_strip = tickets["matricula_parquimetro"].map(normalize_matricula_strip_dot_zero)
    parq_strip = parquimetros["matricula"].map(normalize_matricula_strip_dot_zero)
    variants.append(("quitar_dot_cero_final", ticket_strip, parq_strip))

    ticket_digits = tickets["matricula_parquimetro"].map(normalize_matricula_digits_only)
    parq_digits = parquimetros["matricula"].map(normalize_matricula_digits_only)
    variants.append(("solo_digitos", ticket_digits, parq_digits))

    digit_lengths = pd.concat([ticket_digits.dropna().astype(str).str.len(), parq_digits.dropna().astype(str).str.len()])
    if not digit_lengths.empty:
        zfill_width = int(digit_lengths.max())
        ticket_zfill = tickets["matricula_parquimetro"].map(lambda value: normalize_matricula_digits_zfill(value, zfill_width))
        parq_zfill = parquimetros["matricula"].map(lambda value: normalize_matricula_digits_zfill(value, zfill_width))
        variants.append((f"solo_digitos_zfill_{zfill_width}", ticket_zfill, parq_zfill))

    rows = []
    for name, ticket_norm, parq_norm in variants:
        ticket_set = set(ticket_norm.dropna())
        parq_set = set(parq_norm.dropna())
        intersection = ticket_set & parq_set
        ticket_non_null = ticket_norm.notna()
        ticket_in_parq = ticket_non_null & ticket_norm.isin(parq_set)
        rows.append({
            "normalizacion_candidata": name,
            "n_matriculas_interseccion": len(intersection),
            "pct_tiques_con_matricula_en_parquimetros": round(ticket_in_parq.sum() / ticket_non_null.sum() * 100, 3) if ticket_non_null.sum() else 0.0,
        })
    return pd.DataFrame(rows)


def temporal_join_tickets_parquimetros(
    tickets: pd.DataFrame,
    parquimetros: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    work = tickets.copy().reset_index(drop=True)
    work["_tique_id"] = range(len(work))
    work["matricula_norm"] = work["matricula_parquimetro"].map(normalize_matricula)
    work["fecha_inicio"] = pd.to_datetime(work["fecha_inicio"], errors="coerce")

    candidates = work.merge(
        parquimetros,
        on="matricula_norm",
        how="left",
        validate="many_to_many",
        suffixes=("", "_parquimetro"),
    )

    has_matricula_tique = work.set_index("_tique_id")["matricula_norm"].notna()
    has_candidate = candidates["_parquimetro_row_id"].notna()
    is_vigente = (
        has_candidate
        & candidates["fecha_de_alta"].notna()
        & (candidates["fecha_inicio"] >= candidates["fecha_de_alta"])
        & (candidates["fecha_de_baja"].isna() | (candidates["fecha_inicio"] < candidates["fecha_de_baja"]))
    )
    candidates["_is_vigente"] = is_vigente

    candidate_counts = candidates.loc[has_candidate].groupby("_tique_id").size()
    vigente_counts = candidates.loc[is_vigente].groupby("_tique_id").size()

    status = work[["_tique_id", "periodo_inicio"]].copy()
    status["has_matricula_tique"] = status["_tique_id"].map(has_matricula_tique).fillna(False).astype(bool)
    status["n_parquimetros_matricula"] = status["_tique_id"].map(candidate_counts).fillna(0).astype(int)
    status["n_parquimetros_vigentes"] = status["_tique_id"].map(vigente_counts).fillna(0).astype(int)

    status["match_status_parquimetro"] = "matched_unique"
    status.loc[~status["has_matricula_tique"], "match_status_parquimetro"] = "no_matricula_tique"
    status.loc[
        status["has_matricula_tique"] & (status["n_parquimetros_matricula"] == 0),
        "match_status_parquimetro",
    ] = "no_parquimetro_matricula"
    status.loc[
        status["has_matricula_tique"]
        & (status["n_parquimetros_matricula"] > 0)
        & (status["n_parquimetros_vigentes"] == 0),
        "match_status_parquimetro",
    ] = "no_vigente_en_fecha"
    status.loc[status["n_parquimetros_vigentes"] > 1, "match_status_parquimetro"] = "ambiguous_multiple_vigente"

    matched_ids = status.loc[status["match_status_parquimetro"] == "matched_unique", "_tique_id"]
    joined = candidates.loc[candidates["_is_vigente"] & candidates["_tique_id"].isin(matched_ids)].copy()

    diagnostic_cols = ["_tique_id", "matricula_norm", "_parquimetro_row_id", "_is_vigente"]
    joined = joined.drop(columns=[col for col in diagnostic_cols if col in joined.columns]).reset_index(drop=True)

    return joined, status, candidates


def coherence_summary(left: pd.Series, right: pd.Series, label: str, normalize: str = "numeric") -> pd.DataFrame:
    if normalize == "numeric":
        left_norm = pd.to_numeric(left, errors="coerce").astype("Int64")
        right_norm = pd.to_numeric(right, errors="coerce").astype("Int64")
    elif normalize == "text":
        left_norm = left.map(normalize_text).astype("string")
        right_norm = right.map(normalize_text).astype("string")
    else:
        left_norm = left.astype("string")
        right_norm = right.astype("string")

    status = pd.Series("coincide", index=left.index)
    status[left_norm.isna()] = "izquierda_nula"
    status[right_norm.isna()] = "derecha_nula"
    status[left_norm.notna() & right_norm.notna() & (left_norm != right_norm)] = "discrepa"

    out = status.value_counts(dropna=False).rename_axis("estado_coherencia").reset_index(name="n_tiques")
    out.insert(0, "comparacion", label)
    out["pct_tiques_matched"] = (out["n_tiques"] / len(status) * 100).round(3) if len(status) else 0.0
    return out


parquimetros_view = load_parquimetros_view(clean_paths["ser_parquimetros"])
parquimetros_schema_check = pd.DataFrame([
    {"columna_parquimetro": col, "presente": col in parquimetros_view.columns}
    for col in [
        "matricula", "fecha_de_alta", "fecha_de_baja", "cod_distrito_parquimetro", "distrito_parquimetro",
        "cod_barrio_parquimetro", "num_barrio_parquimetro", "barrio_parquimetro", "calle", "gis_x", "gis_y", "longitud", "latitud",
    ]
])

tickets_matricula_diag = tiques_after_duration_sample.copy()
tickets_matricula_diag["matricula_norm"] = tickets_matricula_diag["matricula_parquimetro"].map(normalize_matricula)

ticket_profile_summary, ticket_profile_examples, ticket_profile_lengths, ticket_profile_top = matricula_profile(
    tickets_matricula_diag, "matricula_parquimetro", "matricula_norm", "tiques_muestra"
)
parq_profile_summary, parq_profile_examples, parq_profile_lengths, parq_profile_top = matricula_profile(
    parquimetros_view, "matricula", "matricula_norm", "parquimetros"
)
matricula_set_summary, top_unmatched_ticket_matriculas = compare_matricula_sets(
    tickets_matricula_diag, parquimetros_view, "matricula_norm", "matricula_norm"
)
matricula_variant_summary = matricula_variant_diagnostics(tiques_after_duration_sample, parquimetros_view)

tiques_join_parquimetro_sample, parquimetro_match_status_sample, _parquimetro_candidates_sample = temporal_join_tickets_parquimetros(
    tiques_after_duration_sample,
    parquimetros_view,
)

match_summary = (
    parquimetro_match_status_sample["match_status_parquimetro"]
    .value_counts(dropna=False)
    .rename_axis("match_status_parquimetro")
    .reset_index(name="n_tiques")
)
match_summary["pct_tiques"] = (match_summary["n_tiques"] / match_summary["n_tiques"].sum() * 100).round(3)

period_join_summary = (
    parquimetro_match_status_sample
    .assign(
        matched_unique=lambda df: df["match_status_parquimetro"].eq("matched_unique"),
        no_parquimetro_matricula=lambda df: df["match_status_parquimetro"].eq("no_parquimetro_matricula"),
        no_vigente_en_fecha=lambda df: df["match_status_parquimetro"].eq("no_vigente_en_fecha"),
        ambiguous_multiple_vigente=lambda df: df["match_status_parquimetro"].eq("ambiguous_multiple_vigente"),
    )
    .groupby("periodo_inicio", as_index=False)
    .agg(
        n_tiques=("_tique_id", "size"),
        n_matched_unique=("matched_unique", "sum"),
        n_no_parquimetro_matricula=("no_parquimetro_matricula", "sum"),
        n_no_vigente_en_fecha=("no_vigente_en_fecha", "sum"),
        n_ambiguous_multiple_vigente=("ambiguous_multiple_vigente", "sum"),
    )
)
period_join_summary["pct_matched_unique"] = (
    period_join_summary["n_matched_unique"] / period_join_summary["n_tiques"] * 100
).round(3)
period_join_summary = period_join_summary[[
    "periodo_inicio", "n_tiques", "n_matched_unique", "pct_matched_unique",
    "n_no_parquimetro_matricula", "n_no_vigente_en_fecha", "n_ambiguous_multiple_vigente",
]]

matched_barrio_examples = tiques_join_parquimetro_sample[[
    "cod_distrito", "distrito", "cod_barrio", "barrio",
    "cod_distrito_parquimetro", "distrito_parquimetro", "cod_barrio_parquimetro",
    "num_barrio_parquimetro", "barrio_parquimetro", "matricula_parquimetro", "matricula",
]].head(20)

barrio_coherence_tables = pd.concat([
    coherence_summary(tiques_join_parquimetro_sample["cod_barrio"], tiques_join_parquimetro_sample["cod_barrio_parquimetro"], "cod_barrio_tique_vs_cod_barrio_parquimetro", "numeric"),
    coherence_summary(tiques_join_parquimetro_sample["cod_barrio"], tiques_join_parquimetro_sample["num_barrio_parquimetro"], "cod_barrio_tique_vs_num_barrio_parquimetro", "numeric"),
    coherence_summary(tiques_join_parquimetro_sample["cod_distrito"], tiques_join_parquimetro_sample["cod_distrito_parquimetro"], "cod_distrito_tique_vs_cod_distrito_parquimetro", "numeric"),
    coherence_summary(tiques_join_parquimetro_sample["barrio"], tiques_join_parquimetro_sample["barrio_parquimetro"], "barrio_norm_tique_vs_barrio_norm_parquimetro", "text"),
], ignore_index=True)

join_contract = pd.DataFrame([
    {"campo_tique": "matricula_parquimetro", "campo_parquimetro": "matricula", "regla": "igualdad normalizada"},
    {"campo_tique": "fecha_inicio", "campo_parquimetro": "fecha_de_alta", "regla": ">= fecha_de_alta"},
    {"campo_tique": "fecha_inicio", "campo_parquimetro": "fecha_de_baja", "regla": "< fecha_de_baja si existe"},
    {"campo_tique": "fecha_inicio", "campo_parquimetro": "fecha_de_baja", "regla": "vigente si fecha_de_baja es nula"},
])

non_linkable_channels = (
    tiques_after_duration_sample.loc[
        tiques_after_duration_sample["matricula_parquimetro"].map(classify_ser_identifier).eq("canal_app_o_digital"),
        "matricula_parquimetro",
    ]
    .map(normalize_matricula)
    .value_counts()
    .head(20)
    .rename_axis("canal_app_no_enlazable")
    .reset_index(name="n_tiques")
)
physical_identifier = tiques_after_duration_sample["matricula_parquimetro"].map(classify_ser_identifier).eq("identificador_fisico_numeric")
n_tiques_fisicos = int(physical_identifier.sum())
n_matched_unique = int(match_summary.loc[match_summary["match_status_parquimetro"].eq("matched_unique"), "n_tiques"].sum())
n_no_vigente = int(match_summary.loc[match_summary["match_status_parquimetro"].eq("no_vigente_en_fecha"), "n_tiques"].sum())
n_sin_inventario = int(n_tiques_fisicos - n_matched_unique - n_no_vigente)
physical_subset_summary = pd.DataFrame([{
    "n_tiques_fisicos": n_tiques_fisicos,
    "n_matched_unique": n_matched_unique,
    "n_no_vigente_en_fecha": n_no_vigente,
    "n_fisicos_sin_inventario": n_sin_inventario,
    "pct_matched_unique_sobre_fisicos": round(n_matched_unique / n_tiques_fisicos * 100, 3) if n_tiques_fisicos else 0.0,
    "pct_no_vigente_sobre_fisicos": round(n_no_vigente / n_tiques_fisicos * 100, 3) if n_tiques_fisicos else 0.0,
    "pct_sin_inventario_sobre_fisicos": round(n_sin_inventario / n_tiques_fisicos * 100, 3) if n_tiques_fisicos else 0.0,
}])
normalization_brief = matricula_variant_summary[[
    "normalizacion_candidata",
    "n_matriculas_interseccion",
    "pct_tiques_con_matricula_en_parquimetros",
]].copy()
territorial_physical_summary = barrio_coherence_tables[
    barrio_coherence_tables["comparacion"].isin([
        "cod_barrio_tique_vs_num_barrio_parquimetro",
        "cod_distrito_tique_vs_cod_distrito_parquimetro",
    ])
].copy()

print("Distribucion app/fisico en tiques depurados de muestra")
display(
    tiques_after_duration_sample["matricula_parquimetro"]
    .map(classify_ser_identifier)
    .value_counts(dropna=False)
    .rename_axis("tipo_identificador_ser")
    .reset_index(name="n_tiques")
    .assign(pct_tiques=lambda df: (df["n_tiques"] / df["n_tiques"].sum() * 100).round(3))
)
print("Top canales/app no enlazables")
display(non_linkable_channels)
print("Cobertura del join parquimetro sobre la muestra")
display(match_summary)
display(physical_subset_summary)
print("Nota: `solo_digitos` evalua identificadores numericos y no convierte pagos app/canal en parquimetros fisicos.")
print("Coherencia territorial del subconjunto fisico enlazado")
display(territorial_physical_summary)


Distribucion app/fisico en tiques depurados de muestra


,tipo_identificador_ser,n_tiques,pct_tiques
0,canal_app_o_digital,1990184,65.041
1,identificador_fisico_numeric,1069700,34.959


Top canales/app no enlazables


,canal_app_no_enlazable,n_tiques
0,APP-MOVIL,659275
1,ELPARKING,433301
2,TELPARK,407114
3,EASYPARK,396968
4,PARKINGLIBRE,71637
5,PARCLICK,9764
6,BIPDRIVE,9435
7,BLINKAY,2676
8,MOWIZ,14


Cobertura del join parquimetro sobre la muestra


,match_status_parquimetro,n_tiques,pct_tiques
0,no_parquimetro_matricula,1990307,65.045
1,matched_unique,1068390,34.916
2,no_vigente_en_fecha,1187,0.039


,n_tiques_fisicos,n_matched_unique,n_no_vigente_en_fecha,n_fisicos_sin_inventario,pct_matched_unique_sobre_fisicos,pct_no_vigente_sobre_fisicos,pct_sin_inventario_sobre_fisicos
0,1069700,1068390,1187,123,99.878,0.111,0.011


Nota: `solo_digitos` evalua identificadores numericos y no convierte pagos app/canal en parquimetros fisicos.
Coherencia territorial del subconjunto fisico enlazado


,comparacion,estado_coherencia,n_tiques,pct_tiques_matched
1,cod_barrio_tique_vs_num_barrio_parquimetro,coincide,1065552,99.734
2,cod_barrio_tique_vs_num_barrio_parquimetro,discrepa,2838,0.266
3,cod_distrito_tique_vs_cod_distrito_parquimetro,coincide,1068390,100.000


**Lectura/decision.** La cobertura global del join parquimetro es baja porque la mayoria de registros de la muestra son pagos app/canal digital y no identificadores fisicos. En cambio, el subconjunto fisico numerico enlaza mayoritariamente con el inventario de parquimetros, con una fraccion menor sin vigencia o sin inventario. Este diagnostico se conserva como evidencia, pero no es el eje del core: barrio sigue siendo la unidad espacial principal.

## 7. Construccion de base barrio SER

El core se construye a escala barrio. Este bloque crea claves canonicas de barrio para los tiques ya filtrados por calendario y duracion normativa, sin depender del join a parquimetro.


In [9]:
def classify_ser_identifier(value: object) -> str:
    text = normalize_matricula(value)
    if pd.isna(text):
        return "sin_identificador"
    if re.search(r"[A-Z]", str(text)):
        return "canal_app_o_digital"
    return "identificador_fisico_numeric"


def resolve_barrio_codes(df: pd.DataFrame, source_name: str, allow_num_barrio: bool = False) -> pd.DataFrame:
    out = df.copy()
    cod_distrito_raw = pd.to_numeric(out.get("cod_distrito"), errors="coerce")
    cod_barrio_raw = pd.to_numeric(out.get("cod_barrio"), errors="coerce")

    if allow_num_barrio and "num_barrio" in out.columns:
        num_barrio_raw = pd.to_numeric(out["num_barrio"], errors="coerce")
    else:
        num_barrio_raw = pd.Series(pd.NA, index=out.index, dtype="Float64")

    has_composite_cod_barrio = cod_barrio_raw.ge(100).fillna(False)
    distrito_from_composite = (cod_barrio_raw // 100).where(has_composite_cod_barrio)
    barrio_from_composite = (cod_barrio_raw % 100).where(has_composite_cod_barrio)

    cod_distrito_resolved = cod_distrito_raw.where(cod_distrito_raw.notna(), distrito_from_composite)
    if allow_num_barrio and num_barrio_raw.notna().any():
        cod_barrio_resolved = num_barrio_raw.where(num_barrio_raw.notna(), barrio_from_composite.where(has_composite_cod_barrio, cod_barrio_raw))
    else:
        cod_barrio_resolved = barrio_from_composite.where(has_composite_cod_barrio, cod_barrio_raw)

    out["cod_distrito"] = cod_distrito_resolved.round().astype("Int64")
    out["cod_barrio"] = cod_barrio_resolved.round().astype("Int64")
    out["cod_barrio_compuesto"] = (out["cod_distrito"] * 100 + out["cod_barrio"]).astype("Int64")
    out["barrio_key"] = (
        out["cod_distrito"].astype("string").str.zfill(2)
        + "_"
        + out["cod_barrio"].astype("string").str.zfill(2)
    )
    valid_key = (
        out["cod_distrito"].notna()
        & out["cod_barrio"].notna()
        & out["cod_distrito"].gt(0)
        & out["cod_barrio"].gt(0)
    )
    out.loc[~valid_key, "barrio_key"] = pd.NA
    out["barrio_norm_diag"] = out["barrio"].map(normalize_text) if "barrio" in out.columns else pd.NA

    diagnostics = pd.DataFrame([{
        "source": source_name,
        "n_registros": int(len(out)),
        "n_cod_distrito_nulo": int(out["cod_distrito"].isna().sum()),
        "n_cod_barrio_nulo": int(out["cod_barrio"].isna().sum()),
        "n_barrio_key_valido": int(out["barrio_key"].notna().sum()),
        "n_barrio_key_invalido": int(out["barrio_key"].isna().sum()),
        "n_barrios_unicos": int(out["barrio_key"].nunique(dropna=True)),
    }])
    return out, diagnostics


def add_barrio_keys(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    out, diagnostics = resolve_barrio_codes(df, "tiques_after_duration_sample", allow_num_barrio=False)
    out["tipo_identificador_ser"] = out["matricula_parquimetro"].map(classify_ser_identifier)
    return out, diagnostics


tiques_barrio_keyed_sample, barrio_key_diagnostics = add_barrio_keys(tiques_after_duration_sample)
tiques_barrio_base_sample = tiques_barrio_keyed_sample.loc[tiques_barrio_keyed_sample["barrio_key"].notna()].reset_index(drop=True)
barrio_base_filter_summary = pd.DataFrame([{
    "n_entrada_tras_duracion": int(len(tiques_after_duration_sample)),
    "n_con_barrio_key_valido": int(tiques_barrio_keyed_sample["barrio_key"].notna().sum()),
    "n_excluidos_barrio_key_invalido": int(tiques_barrio_keyed_sample["barrio_key"].isna().sum()),
    "n_salida_base_barrio": int(len(tiques_barrio_base_sample)),
}])

barrio_period_distribution = (
    tiques_barrio_base_sample.groupby("periodo_inicio", dropna=False, as_index=False)
    .agg(
        n_registros=("matricula_parquimetro", "size"),
        n_barrios_unicos=("barrio_key", "nunique"),
    )
)

display(barrio_base_filter_summary)
display(barrio_period_distribution)
print(f"Columnas finales previstas para tiques barrio: {len(FINAL_BARRIO_TICKETS_COLUMNS)}")


,n_entrada_tras_duracion,n_con_barrio_key_valido,n_excluidos_barrio_key_invalido,n_salida_base_barrio
0,3059884,3059882,2,3059882


,periodo_inicio,n_registros,n_barrios_unicos
0,2023_q1,236666,55
1,2023_q2,235778,55
2,2023_q3,234952,60
3,2023_q4,237354,60
4,2024_q1,237262,60
5,2024_q2,235182,60
6,2024_q3,235878,60
7,2024_q4,235501,63
8,2025_q1,235698,63
9,2025_q2,234125,63


Columnas finales previstas para tiques barrio: 19


**Lectura/decision.** La base barrio se construye directamente desde los tiques depurados por calendario y duracion, usando codigos territoriales del propio tique. En este bloque se aplica explicitamente la regla de conservar solo registros con `barrio_key` valido; los registros sin clave territorial operativa quedan fuera de la base barrio. La distribucion app/fisico ya se documenta en el bloque 6, por lo que aqui solo se conservan controles de cobertura territorial.


## 8. Capacidad anual por barrio desde `ser_calles_plazas`

La capacidad estructural del core se agrega por barrio y año desde `ser_calles_plazas`. La funcion resuelve primero la codificacion de barrio de la fuente de capacidad y solo agrega si la clave barrio puede construirse de forma inequivoca.


In [10]:
def compute_barrio_capacity_anio(calles_plazas: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    required = {"anio", "cod_distrito", "cod_barrio", "barrio", "calle", "numero_plazas"}
    missing = sorted(required.difference(calles_plazas.columns))
    if missing:
        raise ValueError(f"No se puede calcular capacidad barrio: faltan columnas {missing}")

    work, key_diag = resolve_barrio_codes(calles_plazas, "ser_calles_plazas", allow_num_barrio=True)
    if work["barrio_key"].isna().any():
        bad = int(work["barrio_key"].isna().sum())
        raise ValueError(f"No se puede resolver barrio_key de forma inequivoca en {bad} registros de ser_calles_plazas.")

    work["numero_plazas"] = pd.to_numeric(work["numero_plazas"], errors="coerce")
    work["calle_norm"] = work["calle"].map(normalize_street)
    work["barrio_norm_diag"] = work["barrio"].map(normalize_text)

    group_cols = ["anio", "cod_distrito", "cod_barrio", "barrio_key", "cod_barrio_compuesto"]
    capacity = (
        work.groupby(group_cols, dropna=False, as_index=False)
        .agg(
            barrio=("barrio", "first"),
            plazas_barrio_anio=("numero_plazas", "sum"),
            n_calles_barrio_anio=("calle_norm", "nunique"),
        )
    )
    capacity = capacity[[
        "anio", "cod_distrito", "cod_barrio", "barrio", "barrio_key",
        "cod_barrio_compuesto", "plazas_barrio_anio", "n_calles_barrio_anio",
    ]]

    if "color" in work.columns:
        color_pivot = (
            work.assign(color_norm=work["color"].map(normalize_text))
            .pivot_table(
                index=["anio", "barrio_key"],
                columns="color_norm",
                values="numero_plazas",
                aggfunc="sum",
                fill_value=0,
            )
            .reset_index()
        )
        color_pivot.columns = [
            str(col) if col in ["anio", "barrio_key"] else "plazas_color_" + re.sub(r"[^A-Z0-9]+", "_", str(col)).strip("_").lower()
            for col in color_pivot.columns
        ]
        capacity = capacity.merge(color_pivot, on=["anio", "barrio_key"], how="left", validate="one_to_one")

    controls = pd.DataFrame([{
        "n_barrios_anio_con_capacidad": int(len(capacity)),
        "anios_presentes": ", ".join(map(str, sorted(capacity["anio"].dropna().unique()))),
        "plazas_totales": int(capacity["plazas_barrio_anio"].sum()),
    }])
    return capacity, key_diag, controls


calles_plazas_clean = pd.read_parquet(clean_paths["ser_calles_plazas"])
barrio_capacity_anio_sample, calles_key_diagnostics, capacity_controls = compute_barrio_capacity_anio(calles_plazas_clean)

capacity_by_year = (
    barrio_capacity_anio_sample.groupby("anio", as_index=False)
    .agg(
        n_barrios=("barrio_key", "nunique"),
        plazas_totales=("plazas_barrio_anio", "sum"),
    )
)


ticket_barrio_anio = (
    tiques_barrio_base_sample.groupby(["anio", "barrio_key"], dropna=False, as_index=False)
    .agg(n_tiques=("matricula_parquimetro", "size"))
)
capacity_barrio_anio_keys = barrio_capacity_anio_sample[["anio", "barrio_key"]].drop_duplicates()

ticket_barrio_anio_keys = set(map(tuple, ticket_barrio_anio[["anio", "barrio_key"]].dropna().to_numpy()))
capacity_barrio_anio_key_set = set(map(tuple, capacity_barrio_anio_keys.dropna().to_numpy()))
tiques_sin_capacidad_keys = ticket_barrio_anio_keys - capacity_barrio_anio_key_set
capacidad_sin_tiques_keys = capacity_barrio_anio_key_set - ticket_barrio_anio_keys

barrio_anio_capacity_coverage = pd.DataFrame([{
    "n_barrio_anio_tiques": len(ticket_barrio_anio_keys),
    "n_barrio_anio_capacidad": len(capacity_barrio_anio_key_set),
    "n_barrio_anio_tiques_sin_capacidad": len(tiques_sin_capacidad_keys),
    "n_barrio_anio_capacidad_sin_tiques": len(capacidad_sin_tiques_keys),
}])

if tiques_sin_capacidad_keys:
    tiques_sin_capacidad_barrio_anio = ticket_barrio_anio[
        ticket_barrio_anio[["anio", "barrio_key"]].apply(tuple, axis=1).isin(tiques_sin_capacidad_keys)
    ].sort_values(["anio", "barrio_key"])
else:
    tiques_sin_capacidad_barrio_anio = pd.DataFrame(columns=["anio", "barrio_key", "n_tiques"])

ticket_barrio_keys = set(tiques_barrio_base_sample["barrio_key"].dropna())
capacity_barrio_keys = set(barrio_capacity_anio_sample["barrio_key"].dropna())
barrios_tiques_sin_capacidad = pd.DataFrame({"barrio_key": sorted(ticket_barrio_keys - capacity_barrio_keys)})
barrios_capacidad_sin_tiques_muestra = pd.DataFrame({"barrio_key": sorted(capacity_barrio_keys - ticket_barrio_keys)})

capacity_contract = pd.DataFrame([
    {"campo": col, "permanece_en_output": True}
    for col in FINAL_BARRIO_CAPACITY_COLUMNS
])
FINAL_BARRIO_CAPACITY_OPTIONAL_COLOR_COLUMNS = [col for col in barrio_capacity_anio_sample.columns if col.startswith("plazas_color_")]

capacity_key_coverage = calles_key_diagnostics[[
    "n_registros", "n_barrio_key_valido", "n_barrio_key_invalido", "n_barrios_unicos"
]].copy()
color_columns_contract = pd.DataFrame({
    "campo_color_opcional": FINAL_BARRIO_CAPACITY_OPTIONAL_COLOR_COLUMNS,
    "se_conserva_si_existe": True,
})

display(capacity_key_coverage)
display(capacity_controls)
display(capacity_by_year)
display(barrio_anio_capacity_coverage)
if tiques_sin_capacidad_barrio_anio.empty:
    print("No hay barrios-año de tiques sin capacidad en la muestra.")
else:
    display(tiques_sin_capacidad_barrio_anio.head(30))
print(f"Columnas finales obligatorias de capacidad barrio: {len(FINAL_BARRIO_CAPACITY_COLUMNS)}")
if not color_columns_contract.empty:
    display(color_columns_contract)


,n_registros,n_barrio_key_valido,n_barrio_key_invalido,n_barrios_unicos
0,134909,134909,0,65


,n_barrios_anio_con_capacidad,anios_presentes,plazas_totales
0,253,"2023, 2024, 2025, 2026",710186


,anio,n_barrios,plazas_totales
0,2023,60,169700
1,2024,63,178018
2,2025,65,181389
3,2026,65,181079


,n_barrio_anio_tiques,n_barrio_anio_capacidad,n_barrio_anio_tiques_sin_capacidad,n_barrio_anio_capacidad_sin_tiques
0,252,253,0,1


No hay barrios-año de tiques sin capacidad en la muestra.
Columnas finales obligatorias de capacidad barrio: 8


,campo_color_opcional,se_conserva_si_existe
0,plazas_color_alta_rotacion,True
1,plazas_color_azul,True
2,plazas_color_naranja,True
3,plazas_color_rojo,True
4,plazas_color_verde,True


**Lectura/decision.** La capacidad anual se resuelve a escala barrio, no a unidad proxy. `ser_calles_plazas` contiene `num_barrio`, que permite construir el numero de barrio dentro del distrito de forma compatible con tiques. Las columnas `plazas_color_*` se declaran como salida opcional y se conservaran solo si existen en la tabla calculada.


## 9. Escritura de outputs

Este bloque declara el plan de escritura y las salvaguardas asociadas. Por defecto no escribe outputs de producción porque `WRITE_PRODUCTION_OUTPUTS=False`. La producción completa ya se ejecutó de forma controlada y fue validada posteriormente; cualquier nueva escritura debe activarse explícitamente.

La salida `ser_tiques_barrio_base` se genera exclusivamente desde el procesamiento completo por particiones, preservando `periodo_inicio` como partición Hive en la ruta y no como columna física. La función `write_production_outputs` queda limitada a la escritura de `ser_barrio_capacidad_anio.parquet`.


In [11]:
def write_production_outputs(tickets_barrio_base: pd.DataFrame | None, capacidad_barrio_anio: pd.DataFrame) -> None:
    """Escribe únicamente la capacidad barrio-año.

    La base `ser_tiques_barrio_base` se escribe por particiones desde
    `process_tickets_barrio_base_parts`. No debe escribirse desde un DataFrame único,
    porque se perdería el diseño físico validado con partición Hive `periodo_inicio=...`.
    """
    if not WRITE_PRODUCTION_OUTPUTS:
        print("WRITE_PRODUCTION_OUTPUTS=False; no se escriben salidas de produccion en esta iteracion.")
        return

    if tickets_barrio_base is not None:
        raise ValueError(
            "ser_tiques_barrio_base debe escribirse exclusivamente por particiones "
            "desde process_tickets_barrio_base_parts; no desde write_production_outputs."
        )

    capacity_output_columns = FINAL_BARRIO_CAPACITY_COLUMNS + [
        col for col in FINAL_BARRIO_CAPACITY_OPTIONAL_COLOR_COLUMNS
        if col in capacidad_barrio_anio.columns
    ]
    missing_capacity = [col for col in FINAL_BARRIO_CAPACITY_COLUMNS if col not in capacidad_barrio_anio.columns]
    if missing_capacity:
        raise ValueError(f"Faltan columnas finales en capacidad_barrio_anio: {missing_capacity}")

    OUTPUT_BARRIO_CAPACITY_PATH.parent.mkdir(parents=True, exist_ok=True)
    capacidad_barrio_anio[capacity_output_columns].to_parquet(OUTPUT_BARRIO_CAPACITY_PATH, index=False)


planned_write_checks = pd.DataFrame([
    {"output": "ser_tiques_barrio_base", "path": relpath(OUTPUT_TICKETS_BARRIO_DIR), "write_enabled": WRITE_PRODUCTION_OUTPUTS},
    {"output": "ser_barrio_capacidad_anio", "path": relpath(OUTPUT_BARRIO_CAPACITY_PATH), "write_enabled": WRITE_PRODUCTION_OUTPUTS},
])
display(planned_write_checks)
if not WRITE_PRODUCTION_OUTPUTS:
    print("No se escriben salidas. La escritura real solo ocurre en la ejecucion completa con PROCESS_TICKETS_FULL=True y WRITE_PRODUCTION_OUTPUTS=True.")


,output,path,write_enabled
0,ser_tiques_barrio_base,data/processed/core/ser/ser_tiques_barrio_base,True
1,ser_barrio_capacidad_anio,data/processed/core/ser/ser_barrio_capacidad_anio.parquet,True


**Lectura/decisión.** Este bloque declara el plan de escritura y las salvaguardas asociadas. La tabla de capacidad anual por barrio se escribe mediante `write_production_outputs`, mientras que `ser_tiques_barrio_base` se genera exclusivamente desde el procesamiento completo por particiones. Esta separación evita escribir los tiques como un único Parquet no particionado y mantiene el diseño físico validado: `periodo_inicio` se conserva como partición Hive en la ruta, no como columna física dentro de cada fichero.


## 10. Preparacion de ejecucion completa por particiones

Este bloque deja preparado el procesamiento completo de `ser_tiques_barrio_base` por particiones. Por defecto no se ejecuta porque `PROCESS_TICKETS_FULL=False` y no se escribe porque `WRITE_PRODUCTION_OUTPUTS=False`.


In [12]:
def aggregate_duration_summaries(duration_summaries: list[pd.DataFrame]) -> pd.DataFrame:
    if not duration_summaries:
        return pd.DataFrame()
    out = pd.concat(duration_summaries, ignore_index=True)
    group_cols = ["tipo_zona", "limite_minutos"]
    out = out.groupby(group_cols, dropna=False, as_index=False)[["n_entrada", "n_capados", "n_eliminados", "n_salida"]].sum()
    out["pct_capado"] = (out["n_capados"] / out["n_entrada"] * 100).round(3)
    out["pct_eliminado"] = (out["n_eliminados"] / out["n_entrada"] * 100).round(3)
    return out


def validate_written_hive_ticket_dataset(output_dir: Path, expected_rows: int | None = None) -> pd.DataFrame:
    parquet_files = sorted(output_dir.glob("periodo_inicio=*/part_*.parquet"))
    physical_schema_issues = []
    for path in parquet_files:
        physical_columns = pq.read_schema(path).names
        if "periodo_inicio" in physical_columns:
            physical_schema_issues.append(relpath(path))

    dataset = ds.dataset(output_dir, format="parquet", partitioning="hive")
    dataset_rows = dataset.count_rows()
    expected_rows_match = pd.NA if expected_rows is None else dataset_rows == expected_rows
    if physical_schema_issues:
        raise ValueError("Los Parquet escritos no deben contener periodo_inicio como columna fisica.")
    if expected_rows is not None and dataset_rows != expected_rows:
        raise ValueError(f"Recuento Hive inesperado: {dataset_rows} filas leidas frente a {expected_rows} esperadas.")

    return pd.DataFrame([{
        "n_parquet_files": len(parquet_files),
        "periodo_inicio_columna_fisica": len(physical_schema_issues),
        "dataset_hive_legible": True,
        "n_rows_dataset_hive": dataset_rows,
        "n_rows_expected": expected_rows,
        "n_rows_expected_match": expected_rows_match,
        "primer_fichero_con_periodo_inicio_fisico": physical_schema_issues[0] if physical_schema_issues else pd.NA,
    }])


def process_tickets_barrio_base_parts(
    manifest: pd.DataFrame,
    calendar: pd.DataFrame,
    output_dir: Path | None = None,
    write_outputs: bool = False,
    max_parts: int | None = None,
) -> dict[str, pd.DataFrame]:
    if write_outputs:
        output_dir = OUTPUT_TICKETS_BARRIO_DIR if output_dir is None else Path(output_dir)
        if output_dir.exists():
            if not OVERWRITE_OUTPUTS:
                raise FileExistsError(
                    f"La salida {output_dir} ya existe. Activa OVERWRITE_OUTPUTS=True antes de sobrescribir."
                )
            shutil.rmtree(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

    parts = manifest.sort_values(["periodo_inicio", "path"]).copy()
    if max_parts is not None:
        parts = parts.head(max_parts).copy()

    calendar_rows = []
    duration_rows = []
    barrio_rows = []
    identifier_rows = []
    period_rows = []
    write_validation_rows = []

    for _, part in parts.iterrows():
        tickets_part = pd.read_parquet(part["path_abspath"], columns=expected_tiques_final_columns)
        n_input = int(len(tickets_part))

        filtered_calendar, calendar_summary = filter_ser_observable_calendar(tickets_part, calendar)
        filtered_calendar["periodo_inicio"] = part["periodo_inicio"]
        filtered_duration, duration_summary = resolve_duration_by_zone(filtered_calendar)
        keyed, barrio_diag = add_barrio_keys(filtered_duration)
        # periodo_inicio se conserva en la ruta de particion Hive, no como columna fisica,
        # para evitar duplicidad al leer con partitioning='hive'.
        output_part = keyed.loc[keyed["barrio_key"].notna(), FINAL_BARRIO_TICKETS_COLUMNS].reset_index(drop=True)

        calendar_summary["periodo_inicio"] = part["periodo_inicio"]
        calendar_summary["path"] = part["path"]
        calendar_rows.append(calendar_summary)

        duration_summary["periodo_inicio"] = part["periodo_inicio"]
        duration_summary["path"] = part["path"]
        duration_rows.append(duration_summary)

        barrio_diag["periodo_inicio"] = part["periodo_inicio"]
        barrio_diag["path"] = part["path"]
        barrio_rows.append(barrio_diag)

        identifier = (
            output_part["tipo_identificador_ser"].value_counts(dropna=False)
            .rename_axis("tipo_identificador_ser")
            .reset_index(name="n_tiques")
        )
        identifier["periodo_inicio"] = part["periodo_inicio"]
        identifier_rows.append(identifier)

        period_rows.append({
            "periodo_inicio": part["periodo_inicio"],
            "path": part["path"],
            "n_entrada_final_clean": n_input,
            "n_salida_calendario": int(len(filtered_calendar)),
            "n_salida_duracion": int(len(filtered_duration)),
            "n_salida_barrio_base": int(len(output_part)),
        })

        if write_outputs:
            rel_part = Path(part["path"]).relative_to(relpath(TIQUES_FINAL_PARTS_DIR))
            out_path = output_dir / rel_part
            out_path.parent.mkdir(parents=True, exist_ok=True)
            output_part.to_parquet(out_path, index=False)

    if write_outputs:
        expected_rows = None if max_parts is not None else int(sum(row["n_salida_barrio_base"] for row in period_rows))
        write_validation_rows.append(validate_written_hive_ticket_dataset(output_dir, expected_rows=expected_rows))

    return {
        "calendar_summary": pd.DataFrame(calendar_rows),
        "duration_summary": aggregate_duration_summaries(duration_rows),
        "barrio_key_summary": pd.concat(barrio_rows, ignore_index=True) if barrio_rows else pd.DataFrame(),
        "identifier_summary": pd.concat(identifier_rows, ignore_index=True) if identifier_rows else pd.DataFrame(),
        "period_summary": pd.DataFrame(period_rows),
        "write_validation": pd.concat(write_validation_rows, ignore_index=True) if write_validation_rows else pd.DataFrame(),
    }


def aggregate_calendar_summary(calendar_summary: pd.DataFrame) -> pd.DataFrame:
    if calendar_summary.empty:
        return pd.DataFrame()
    sum_cols = [
        "n_tiques_entrada", "n_fecha_sin_calendario", "n_fuera_dia_servicio",
        "n_fuera_horario_lunes_viernes", "n_fuera_horario_sabado", "n_fuera_horario_agosto",
        "n_fuera_horario_24_31_dic", "n_tiques_salida",
    ]
    out = calendar_summary[sum_cols].sum().to_frame().T
    out["pct_eliminado"] = ((1 - out["n_tiques_salida"] / out["n_tiques_entrada"]) * 100).round(3)
    return out


def aggregate_barrio_key_summary(barrio_summary: pd.DataFrame) -> pd.DataFrame:
    if barrio_summary.empty:
        return pd.DataFrame()
    sum_cols = ["n_registros", "n_barrio_key_valido", "n_barrio_key_invalido"]
    out = barrio_summary[sum_cols].sum().to_frame().T
    out["n_barrios_unicos_max_parte"] = barrio_summary["n_barrios_unicos"].max()
    return out


def aggregate_identifier_summary(identifier_summary: pd.DataFrame) -> pd.DataFrame:
    if identifier_summary.empty:
        return pd.DataFrame()
    out = identifier_summary.groupby("tipo_identificador_ser", as_index=False)["n_tiques"].sum()
    out["pct_tiques"] = (out["n_tiques"] / out["n_tiques"].sum() * 100).round(3)
    return out


full_run_plan = pd.DataFrame([{
    "PROCESS_TICKETS_FULL": PROCESS_TICKETS_FULL,
    "WRITE_PRODUCTION_OUTPUTS": WRITE_PRODUCTION_OUTPUTS,
    "OVERWRITE_OUTPUTS": OVERWRITE_OUTPUTS,
    "MAX_PARTS_FULL_RUN": MAX_PARTS_FULL_RUN,
    "SMOKE_TEST_N_PARTS": SMOKE_TEST_N_PARTS,
    "output_dir": relpath(OUTPUT_TICKETS_BARRIO_DIR),
}])
display(full_run_plan)

if PROCESS_TICKETS_FULL:
    parts_limit = SMOKE_TEST_N_PARTS if SMOKE_TEST_N_PARTS is not None else MAX_PARTS_FULL_RUN
    full_run_summaries = process_tickets_barrio_base_parts(
        tiques_final_manifest,
        calendar_ser,
        output_dir=OUTPUT_TICKETS_BARRIO_DIR,
        write_outputs=WRITE_PRODUCTION_OUTPUTS,
        max_parts=parts_limit,
    )
    period_total_summary = (
        full_run_summaries["period_summary"]
        .groupby("periodo_inicio", as_index=False)[[
            "n_entrada_final_clean", "n_salida_calendario", "n_salida_duracion", "n_salida_barrio_base"
        ]]
        .sum()
    )
    display(full_run_summaries["period_summary"].head())
    display(period_total_summary)
    display(aggregate_calendar_summary(full_run_summaries["calendar_summary"]))
    display(full_run_summaries["duration_summary"])
    display(aggregate_barrio_key_summary(full_run_summaries["barrio_key_summary"]))
    display(aggregate_identifier_summary(full_run_summaries["identifier_summary"]))
    if not full_run_summaries["write_validation"].empty:
        display(full_run_summaries["write_validation"])
    if WRITE_PRODUCTION_OUTPUTS:
        write_production_outputs(None, barrio_capacity_anio_sample)
else:
    print("Ejecucion completa preparada pero no activada: PROCESS_TICKETS_FULL=False.")
    if WRITE_PRODUCTION_OUTPUTS:
        print("WRITE_PRODUCTION_OUTPUTS=True no tendra efecto sobre tiques mientras PROCESS_TICKETS_FULL=False.")


,PROCESS_TICKETS_FULL,WRITE_PRODUCTION_OUTPUTS,OVERWRITE_OUTPUTS,MAX_PARTS_FULL_RUN,SMOKE_TEST_N_PARTS,output_dir
0,True,True,False,None,None,data/processed/core/ser/ser_tiques_barrio_base


,periodo_inicio,path,n_entrada_final_clean,n_salida_calendario,n_salida_duracion,n_salida_barrio_base
0,2023_q1,data/interim/ser/ser_tiques/final_clean_parts/periodo_inicio=2023_q1/part_000000__raw_2023_q1.parquet,249976,249973,236923,236923
1,2023_q1,data/interim/ser/ser_tiques/final_clean_parts/periodo_inicio=2023_q1/part_000002__raw_2023_q1.parquet,249978,249976,237046,237046
2,2023_q1,data/interim/ser/ser_tiques/final_clean_parts/periodo_inicio=2023_q1/part_000004__raw_2023_q1.parquet,249974,249973,237233,237233
3,2023_q1,data/interim/ser/ser_tiques/final_clean_parts/periodo_inicio=2023_q1/part_000006__raw_2023_q1.parquet,249966,249964,236939,236939
4,2023_q1,data/interim/ser/ser_tiques/final_clean_parts/periodo_inicio=2023_q1/part_000008__raw_2023_q1.parquet,249976,249973,236680,236680


,periodo_inicio,n_entrada_final_clean,n_salida_calendario,n_salida_duracion,n_salida_barrio_base
0,2023_q1,11907716,11907152,11287870,11287868
1,2023_q2,11665684,11665658,10979735,10979735
2,2023_q3,9651162,9651053,9075663,9075663
3,2023_q4,11560293,11560277,10971316,10971316
4,2024_q1,12104937,12104854,11492457,11492454
5,2024_q2,12490394,12490140,11757135,11757129
6,2024_q3,7910433,7910397,7385137,7385136
7,2024_q4,12593853,12589143,11863161,11863151
8,2025_q1,12749010,12748991,12006838,12006838
9,2025_q2,12361188,12361175,11582448,11582448


,n_tiques_entrada,n_fecha_sin_calendario,n_fuera_dia_servicio,n_fuera_horario_lunes_viernes,n_fuera_horario_sabado,n_fuera_horario_agosto,n_fuera_horario_24_31_dic,n_tiques_salida,pct_eliminado
0,150137884,0,747,61,245,174,4700,150131957,0.004


,tipo_zona,limite_minutos,n_entrada,n_capados,n_eliminados,n_salida,pct_capado,pct_eliminado
0,ALTA ROTACION,45,586051,0,17976,568075,0.000,3.067
1,AZUL,240,57743450,0,4155464,53587986,0.000,7.196
2,AZUL SANITARIA,240,925971,0,33182,892789,0.000,3.583
3,COMERCIALES,480,8638084,927,1,8638083,0.011,0.000
4,TALLERES,300,71918,11,0,71918,0.015,0.000
5,USO DISUASORIO,720,1554051,0,55548,1498503,0.000,3.574
6,VERDE,120,80612432,0,4583249,76029183,0.000,5.686


,n_registros,n_barrio_key_valido,n_barrio_key_invalido,n_barrios_unicos_max_parte
0,141286537,141286515,22,65


,tipo_identificador_ser,n_tiques,pct_tiques
0,canal_app_o_digital,88218516,62.439
1,identificador_fisico_numeric,53067999,37.561


,n_parquet_files,periodo_inicio_columna_fisica,dataset_hive_legible,n_rows_dataset_hive,n_rows_expected,n_rows_expected_match,primer_fichero_con_periodo_inicio_fisico
0,1002,0,True,141286515,141286515,True,<NA>


## 11. Lectura metodológica final y límites

El notebook `04_01_ser_joins_base.ipynb` consolida la base nuclear del bloque SER a escala barrio. Parte de `final_clean_parts`, aplica régimen SER observable, resuelve la duración normativa por `tipo_zona`, construye claves canónicas de barrio para los tiques y calcula la capacidad anual por barrio a partir de `ser_calles_plazas`.

La ejecución completa se realizó sobre 1002 particiones de tiques SER entre 2023 y 2026. A partir de 150.137.884 registros de entrada, el filtro de calendario deja 150.131.957 registros; la depuración por duración normativa deja 141.286.537; y la exigencia de `barrio_key` válido produce una salida final de 141.286.515 registros. La tabla `ser_barrio_capacidad_anio.parquet` contiene 253 combinaciones barrio-año y conserva, cuando existen, las columnas opcionales `plazas_color_*`.

La decisión metodológica central es mantener barrio como unidad espacial robusta del core SER. Esta escala cubre tanto tiques con identificador físico como pagos app/canal digital. El diagnóstico de parquímetros muestra que el subconjunto físico enlaza mayoritariamente con el inventario, pero una parte sustancial del histórico corresponde a canales app o digitales que no pueden imputarse de forma verificable a parquímetro, calle o tramo con las fuentes públicas disponibles. Por tanto, calle/parquímetro queda como análisis complementario posterior limitado a tiques con identificador físico, no como base del proxy principal.

La salida física `ser_tiques_barrio_base` queda organizada como dataset Parquet particionado por carpetas `periodo_inicio=YYYY_qX`. La variable `periodo_inicio` se conserva en la ruta de partición Hive y no dentro de cada Parquet, para evitar duplicidad de esquema al leer con `pyarrow.dataset(..., partitioning="hive")`. La validación posterior confirma que el dataset contiene 1002 ficheros y 141.286.515 filas legibles con el entorno `tfm-parking`.

El notebook queda por defecto en modo seguro: `PROCESS_TICKETS_FULL=False`, `WRITE_PRODUCTION_OUTPUTS=False` y `OVERWRITE_OUTPUTS=False`. Esto evita reescrituras accidentales. Una nueva regeneración completa solo debe realizarse si se activan explícitamente las banderas y se revisa después la validación Hive.

Límites de esta iteración:

- no se construye todavía `SER_barrio_intervalo`;
- no se calcula todavía el target proxy de dificultad;
- no se entrenan modelos;
- no se implementa la escala fina calle/parquímetro;
- no se imputan pagos app/canal digital a ubicaciones físicas;
- la dificultad de aparcar sigue siendo una variable proxy que se construirá en el notebook posterior.

El siguiente paso metodológico es `04_02_ser_panel_barrio_intervalo.ipynb`, donde se agregará `ser_tiques_barrio_base` sobre una malla común barrio-intervalo y se compararán granularidades de 15, 30, 45 y 60 minutos antes de fijar la granularidad temporal final.


### Nota sobre la ejecución conservada con outputs

Esta versión del notebook se ha guardado con outputs visibles tras una ejecución completa controlada, con el objetivo de dejar trazabilidad de los resultados intermedios y globales obtenidos durante la construcción de `ser_tiques_barrio_base`.

La regeneración se realizó tras mover temporalmente los outputs existentes a una carpeta de respaldo. La ejecución produjo de nuevo `ser_tiques_barrio_base` y `ser_barrio_capacidad_anio.parquet` sin activar sobrescritura automática (`OVERWRITE_OUTPUTS=False`). Posteriormente se compararon las salidas nuevas con el respaldo anterior:

- `ser_tiques_barrio_base`: 141.286.515 filas en la versión nueva y 141.286.515 filas en la versión anterior.
- `ser_barrio_capacidad_anio.parquet`: 253 filas en la versión nueva y 253 filas en la versión anterior.
- El tamaño físico de `ser_tiques_barrio_base` fue equivalente en ambas versiones: 3,7G.
- Las particiones `periodo_inicio=YYYY_qX` coincidieron entre ambas versiones.

Tras confirmar la equivalencia, la carpeta de respaldo fue eliminada. Por tanto, los outputs visibles del notebook documentan una ejecución completa reproducible y coherente con los outputs finales conservados en `data/processed/core/ser/`.

Para evitar reescrituras accidentales, cualquier nueva ejecución completa debe hacerse de forma explícita y controlada, revisando previamente las banderas `PROCESS_TICKETS_FULL`, `WRITE_PRODUCTION_OUTPUTS` y `OVERWRITE_OUTPUTS`.